In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from BaselineRemoval import BaselineRemoval
import matplotlib.pyplot as plt
import glob

In [ ]:
data_path = './ML_demo_data/Control/*.txt'
save_path = './ML_demo_data/Control/Control- No Processing.xlsx'

In [69]:
def no_processing(data_path, save_path, delim="\t"):    
    l = [pd.read_csv(filename,header=None,delimiter = "\t") for filename in glob.glob(data_path)]
    data = pd.concat(l, ignore_index=True)
    data.columns = ["positionX", "positionY", "Wavenumber", "Counts"]
    data.drop(data.columns[[0, 1]], axis = 1, inplace = True)

    n =1300
    data_separate = pd.DataFrame(np.hstack(np.vsplit(data.values, len(data) // n)))
    data_separate.columns = data_separate.columns.map(lambda c: chr(c + ord('A')))
    col_to_drop = data_separate.columns[np.array([i for i in range(data_separate.shape[1]) if i != 0 and i%2 != 1])]
    rslt_df = data_separate.drop(col_to_drop, axis=1)
    rslt_df.rename(columns = {'A':'Wavenumber'}, inplace = True)


    #output
    new_df = rslt_df['Wavenumber']
    a = new_df.to_frame(name='Wavenumber')
    # frame = pd.concat([a,rslt_df], axis = 1)
    # frame.to_excel(save_path, index=False)
    rslt_df.to_excel(save_path, index=False)
    
    return rslt_df

In [117]:
data_folders = ["0.2mg", "4mg", "10mg", "Control"]

combined = pd.DataFrame()

for folder in data_folders:
    data_path = f'./ML_demo_data/{folder}/*.txt'
    save_path = f'./ML_demo_data/{folder}/{folder}- No Processing.xlsx'
    frame = no_processing(data_path, save_path)
    frame = frame.T
    if folder == "0.2mg":
        new_cols = frame.loc['Wavenumber'].to_list()
    new_index = [f'{folder}' for i in range(frame.shape[0])]
    frame.insert(0, 'Group', new_index)
    frame.reset_index(drop=True, inplace=True)
    frame = frame[1:]
    combined = pd.concat([combined, frame], axis=0, ignore_index=True)

In [120]:
combined.columns = ["Group"] + new_cols

combined.to_excel("./Combined_No_Preprocessing.xlsx", index=False)